# Poisson RNA profile enhancer–gene evaluation

Epoch 35 (`35-val_loss=0.72645.ckpt`). Accessibility is divided by 100 over the enhancer plus 100 bp on each side. The primary readout is the full, strand-matched gene body, chosen after checking raw RNA coverage. Terminal 10/50/100 kb and TSS scores are controls. All heads use identical enhancer perturbations.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown
RUN = Path('/data1/lesliec/sarthak/data/joint_playground/e2g/k562_poisson_ep35_profile_readout_20260922')


In [ ]:
display(Markdown((RUN / 'coverage_audit/README.md').read_text()))
display(Image(filename=str(RUN / 'coverage_audit/terminal_coverage.png')))


## Run status

The job saves progress and resumable predictions while running. Metrics and plots are generated automatically when all 3,914 in-context pairs are scored. Primary comparisons exclude 39 pairs with incomplete gene bodies.


In [ ]:
status_file = RUN / 'status.json'
print(json.dumps(json.loads(status_file.read_text()), indent=2) if status_file.exists() else 'Waiting for the submitted GPU job to start.')


## Enhancer–gene results

Native profile scores are log2 ratios of summed predictions in the `sqrt(1+coverage)-1` training scale, with a pseudocount of 1. Reconstructed scores invert each base as `p*(p+2)` before summation; they are approximate raw coverage changes. Scalar scores are differences in `log2(1+CPM)`. Negative changes are the expected direction for enhancer reduction.


In [ ]:
if (RUN / 'metrics.tsv').exists():
    metrics = pd.read_csv(RUN / 'metrics.tsv', sep='\t')
    display(metrics[metrics.subset.eq('common_full_gene')])
    display(Image(filename=str(RUN / 'precision_recall.png')))
    display(Markdown((RUN / 'README.md').read_text()))
else:
    print('Final metrics are written automatically after inference completes.')


In [ ]:
if (RUN / 'metrics.tsv').exists():
    display(metrics[metrics.direction.eq('decrease')].pivot(index='readout', columns='subset', values='auprc'))
    print((RUN / 'summary.json').read_text())
